In [ ]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import holoviews as hv
import hvplot.pandas
import numpy as np

from workspace_dani.data_preprocessing.utils import data_handling

hv.extension('bokeh')


In [ ]:
project_root_path = Path.cwd().parents[2]  #/ Path("data_preprocessing") / Path("utils")

In [ ]:
# labels_path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") / Path("invalid") / Path(
#     "sepsis_data_15min.parquet")

labels_path = (project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path("sepsis") /
                Path("valid") / Path(
             "sepsis_data_1h.parquet"))


In [ ]:
df_sepsis = data_handling.load_parquet_table(parquet_path=labels_path, debug_mode=False, backend="numpy_nullable")

In [ ]:
df_sepsis.head()

this is wrong:
df_sepsis["sepsis_offset"] = df_sepsis[["labresultoffset", "abx_time", "samp_time"]].max(axis=1)

In [ ]:

df_sepsis["sepsis_offset"] = df_sepsis["labresultoffset"]


In [ ]:
# df_sepsis["timedelta"] = df_sepsis.apply(lambda x: abs(abs(x["labresultoffset"]) - abs(x["abx_time"])) if pd.isnull(x["samp_time"]) else abs(abs(x["labresultoffset"]) - abs(x["samp_time"])), axis=1)

offsets_max_abs = abs(df_sepsis[["labresultoffset", "abx_time", "samp_time"]].max(axis=1))
offsets_min_abs = abs(df_sepsis[["labresultoffset", "abx_time", "samp_time"]].min(axis=1))

df_sepsis["time_delta"] = abs(offsets_max_abs - offsets_min_abs)


Renaming labresultoffset to "sepsis_offset" for less ambiguity (the labresultoffset is from the SOFA score, which determines the exact timing of sepsis-3 labels)

In [ ]:
df_sepsis_concise = df_sepsis[["patientunitstayid", "sep3", "sepsis_offset", "time_delta"]]

In [ ]:
df_sepsis_concise.head()

### We will just consider sepsis labels that fall within a time frame of 168 hours and explicitly discard sepsis labels after 168 h


In [ ]:
sepsis_exclusion_upper_bound = df_sepsis_concise["sepsis_offset"] > pd.to_timedelta(168, unit="hours")
sepsis_exclusion_lower_bound = df_sepsis_concise["sepsis_offset"] <= pd.to_timedelta(4, unit="hours")

df_sepsis_cohort_excluded_upper = df_sepsis_concise[sepsis_exclusion_upper_bound]
df_sepsis_cohort_excluded_lower = df_sepsis_concise[sepsis_exclusion_lower_bound]

df_sepsis_cohort_included = df_sepsis_concise[df_sepsis_concise["sepsis_offset"] <= pd.to_timedelta(168, unit="hours")]
df_sepsis_cohort_excluded = df_sepsis_concise[sepsis_exclusion_lower_bound ^ sepsis_exclusion_upper_bound]


In [ ]:
df_sepsis_cohort_included.describe()

In [ ]:
df_sepsis_cohort_excluded.describe()

-> Works as intended !

In [ ]:
patient_ids_dir: Path = project_root_path / Path("intermediary_data") / Path("patient_unit_stay_ids") / Path(
    "cohorts") / Path("sepsis")

ids_included_sepsis = df_sepsis_cohort_included["patientunitstayid"].unique().tolist()
df_included_ids_sepsis: pd.DataFrame = pd.DataFrame(data=ids_included_sepsis, columns=["Included ID"])
df_included_ids_sepsis.to_parquet(patient_ids_dir / Path("included_ids_sepsis.parquet"), index=False)

In [ ]:

ids_excluded_sepsis = df_sepsis_cohort_excluded["patientunitstayid"].unique().tolist()
df_excluded_ids_sepsis: pd.DataFrame = pd.DataFrame(data=ids_excluded_sepsis, columns=["Excluded ID"])
df_excluded_ids_sepsis.to_parquet(patient_ids_dir / Path("excluded_ids_sepsis.parquet"), index=False)


In [ ]:
sepsis_outcome_dir: Path = project_root_path / Path("intermediary_data") / Path("outcome_labels") / Path(
    "sepsis")

df_sepsis_concise.to_parquet(sepsis_outcome_dir / Path("sepsis_labels_concise.parquet"), index=False)